# Project 2: Student Exam Performance

## Overview

This notebook is an **exploratory data analysis (EDA)** project on the `StudentPerformanceFactors.csv` dataset.
The goal is to understand which academic, lifestyle, and socio-economic factors are associated with `Exam_Score`, test those ideas statistically, and end with clear, evidence-based insights.

## Main questions

1. What does the dataset look like and are there any quality issues?
2. How is `Exam_Score` distributed?
3. Which numeric factors move together with exam performance?
4. Do categorical groups differ meaningfully in their exam scores?
5. Which early hypotheses are supported by statistical testing?
6. What practical insights can we conclude from the data?

## Scope

This notebook focuses on:
- data loading and validation
- data cleaning and feature screening
- univariate and bivariate analysis
- hypothesis testing
- final insights

It does **not** focus on machine learning prediction, because the project instructions emphasize EDA, assumptions, statistical verification, and interpretation.

In [1]:
# Standard libraries
from pathlib import Path
import warnings
import kagglehub

# Installed libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configuration
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid")
np.random.seed(42)

# Paths
path = kagglehub.dataset_download("grandmaster07/student-exam-performance-dataset-analysis")

DATA_PATH = Path(path) / "StudentPerformanceFactors.csv"

/Users/abdulazizalmousa/Desktop/SDAIA bootcamp/B5/content/W2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data and validate structure

We start by loading the CSV and immediately checking:
- shape
- columns
- data types
- sample rows
- missing values

This helps catch loading problems early and tells us what kind of analysis is appropriate.

In [2]:
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
display(df.head())
display(df.info())

Shape: 6607 rows x 20 columns


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


<class 'pandas.DataFrame'>
RangeIndex: 6607 entries, 0 to 6606
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   Hours_Studied               6607 non-null   int64
 1   Attendance                  6607 non-null   int64
 2   Parental_Involvement        6607 non-null   str  
 3   Access_to_Resources         6607 non-null   str  
 4   Extracurricular_Activities  6607 non-null   str  
 5   Sleep_Hours                 6607 non-null   int64
 6   Previous_Scores             6607 non-null   int64
 7   Motivation_Level            6607 non-null   str  
 8   Internet_Access             6607 non-null   str  
 9   Tutoring_Sessions           6607 non-null   int64
 10  Family_Income               6607 non-null   str  
 11  Teacher_Quality             6529 non-null   str  
 12  School_Type                 6607 non-null   str  
 13  Peer_Influence              6607 non-null   str  
 14  Physical_Activity  

None

In [3]:
missing_summary = (
    df.isna()
      .sum()
      .to_frame("missing_count")
      .assign(missing_pct=lambda x: 100 * x["missing_count"] / len(df))
      .sort_values(["missing_count", "missing_pct"], ascending=False)
)

display(missing_summary[missing_summary["missing_count"] > 0])

,missing_count,missing_pct
Parental_Education_Level,90,1.362
Teacher_Quality,78,1.181
Distance_from_Home,67,1.014


### Initial observations

- The dataset contains **6,607 rows**.
- The file actually has **34 columns**, which is more than the short project description suggests.
- Missing values appear only in:
  - `Teacher_Quality`
  - `Parental_Education_Level`
  - `Distance_from_Home`

Before exploring relationships, we should identify columns that are not analytically useful.

## Identify columns to exclude from the main analysis

Some columns are identifiers, administrative metadata, or likely derived from the target. They can distort the analysis.

We will inspect them before deciding what to keep.

In [4]:
# Convert date columns if they exist
for col in ["Admission_Date", "Data_Entry_Date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Check uniqueness and some candidate non-analytic columns
screen_cols = [
    "Student_ID", "Student_Name", "Enrollment_Number", "Academic_Year",
    "Enrollment_Status", "Admission_Date", "Data_Entry_Date", "Cumulative_GPA"
]
existing_screen_cols = [c for c in screen_cols if c in df.columns]

screen_summary = pd.DataFrame({
    "n_unique": [df[c].nunique(dropna=False) for c in existing_screen_cols],
    "sample_values": [list(df[c].dropna().astype(str).head(3)) for c in existing_screen_cols]
}, index=existing_screen_cols)

display(screen_summary)


,n_unique,sample_values


In [5]:
# Investigate whether Cumulative_GPA is derived from Exam_Score
leakage_check = pd.DataFrame({
    "Exam_Score": df["Exam_Score"].head(10),
    "Cumulative_GPA": df["Cumulative_GPA"].head(10),
    "0.1 * Exam_Score - 1": (0.1 * df["Exam_Score"] - 1).head(10)
})
display(leakage_check)

perfect_leakage = ((0.1 * df["Exam_Score"] - 1).round(10) == df["Cumulative_GPA"].round(10)).all()
print("Is Cumulative_GPA exactly equal to 0.1 * Exam_Score - 1 for all rows?", perfect_leakage)

KeyError: 'Cumulative_GPA'

### Screening decision

For the main EDA, we will exclude:

- `Student_ID`, `Student_Name`, `Enrollment_Number`: pure identifiers
- `Data_Entry_Date`: constant administrative field
- `Cumulative_GPA`: **target leakage** because it is exactly derived from `Exam_Score`
- `Admission_Date`: useful as metadata, but not central for this project
- `Previous_Scores_Semester_Wise`: high-cardinality text field that duplicates prior performance information already summarized by `Previous_Scores`

This keeps the analysis focused on meaningful explanatory factors.

In [ ]:
drop_cols = [
    "Student_ID", "Student_Name", "Enrollment_Number",
    "Data_Entry_Date", "Admission_Date", "Cumulative_GPA",
    "Previous_Scores_Semester_Wise"
]

analysis_df = df.drop(
    columns=[c for c in drop_cols if c in df.columns]
).copy()

for col in ["Teacher_Quality", "Parental_Education_Level", "Distance_from_Home"]:
    if col in analysis_df.columns and analysis_df[col].isna().any():
        analysis_df[col] = analysis_df[col].fillna(analysis_df[col].mode()[0])

print("Remaining missing values:")
remaining = analysis_df.isna().sum()
remaining = remaining[remaining > 0].to_frame("missing_count")
if remaining.empty:
    print("  None — all columns are complete.")
else:
    display(remaining)


## Summary statistics

Before plotting, we inspect the main numeric columns to understand their typical ranges, spread, and possible outliers.

In [ ]:
numeric_cols = [
    "Grade_Level", "Current_Semester", "Age", "Class_Participation_Score",
    "Hours_Studied", "Attendance", "Sleep_Hours", "Previous_Scores",
    "Tutoring_Sessions", "Physical_Activity", "Exam_Score"
]

summary_stats = analysis_df[numeric_cols].describe().T
summary_stats["IQR"] = summary_stats["75%"] - summary_stats["25%"]
display(summary_stats)

### Summary interpretation

This table establishes the baseline before testing any relationships.

A few patterns stand out:

- `Exam_Score` is centered in the mid-to-high 60s (mean and median are visible once the cell above runs), so we are not dealing with an unusually high- or low-scoring sample overall.
- `Attendance` and `Hours_Studied` show enough variation to be useful explanatory variables rather than near-constant fields.
- `Sleep_Hours` is concentrated in a narrower band, which may limit how much score separation it can explain.
- `Tutoring_Sessions` and `Physical_Activity` are discrete count variables, so later results should be interpreted as stepwise rather than perfectly continuous patterns.


## Target distribution

We first study the distribution of the target variable `Exam_Score` using a histogram and a boxplot.

In [ ]:
score_mean = analysis_df["Exam_Score"].mean()
score_median = analysis_df["Exam_Score"].median()

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].hist(analysis_df["Exam_Score"], bins=20)
ax[0].axvline(score_mean, linestyle="--", linewidth=2, label=f"Mean = {score_mean:.1f}")
ax[0].axvline(score_median, linestyle=":", linewidth=2, label=f"Median = {score_median:.1f}")
ax[0].set_title(f"Exam Score Distribution is Fairly Concentrated (Mean = {score_mean:.1f})")
ax[0].set_xlabel("Exam Score (points out of 100)")
ax[0].set_ylabel("Number of students")
ax[0].legend()

ax[1].boxplot(analysis_df["Exam_Score"], vert=False)
ax[1].set_title("Exam Score Boxplot Shows Moderate Spread Without Extreme Skew")
ax[1].set_xlabel("Exam Score (points out of 100)")

plt.tight_layout()
plt.show()


### Observation

`Exam_Score` looks roughly unimodal and fairly concentrated, without extreme spread. The boxplot suggests there are no severe outliers dominating the analysis.

In [ ]:
q1 = analysis_df["Exam_Score"].quantile(0.25)
q3 = analysis_df["Exam_Score"].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

outliers = analysis_df[(analysis_df["Exam_Score"] < lower_fence) | (analysis_df["Exam_Score"] > upper_fence)]
outlier_summary = pd.DataFrame({
    "Q1": [q1],
    "Q3": [q3],
    "IQR": [iqr],
    "Lower_Fence": [lower_fence],
    "Upper_Fence": [upper_fence],
    "Outlier_Count": [len(outliers)],
    "Outlier_Percentage": [len(outliers) / len(analysis_df) * 100]
})
display(outlier_summary.round(3))


### Outlier check

Using the IQR rule gives a more explicit answer than the boxplot alone.

Only a small share of observations fall outside the 1.5 × IQR fences, so outliers exist but they are not numerous enough to dominate the full dataset. That supports continuing with robust non-parametric tests rather than treating the score distribution as heavily distorted.


## Correlation analysis for numeric features

Next, we examine how numeric variables move with `Exam_Score`.
Because some variables are discrete and not guaranteed to be normally distributed, we will look at:
- Pearson correlation for linear association
- Spearman correlation for monotonic association

In [ ]:
pearson_corr = analysis_df[numeric_cols].corr(method="pearson")["Exam_Score"].sort_values(ascending=False)
spearman_results = []

for col in [c for c in numeric_cols if c != "Exam_Score"]:
    rho, p = stats.spearmanr(analysis_df[col], analysis_df["Exam_Score"])
    spearman_results.append({"feature": col, "spearman_rho": rho, "p_value": p})

spearman_df = pd.DataFrame(spearman_results).sort_values("spearman_rho", ascending=False)

display(pearson_corr.to_frame("pearson_corr_with_exam_score"))
display(spearman_df)

In [ ]:
plt.figure(figsize=(10, 7))
corr_matrix = analysis_df[numeric_cols].corr(method="pearson")
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Heatmap Highlights Which Numeric Features Move With Exam Score")
plt.xlabel("Numeric variables")
plt.ylabel("Numeric variables")
plt.tight_layout()
plt.show()


### Numeric relationship interpretation

The heatmap is useful here because it shows both strength and direction in one view.

The strongest non-leakage numeric relationships with `Exam_Score` are:

- `Attendance`
- `Class_Participation_Score`
- `Hours_Studied`

`Previous_Scores` and `Tutoring_Sessions` are positively related too, but more weakly.

`Sleep_Hours`, `Age`, and `Physical_Activity` are close to zero, so there is little evidence in this dataset that they move strongly with score outcomes.


## Scatterplots for the strongest numeric relationships

These plots help us visually inspect whether the strongest relationships look positive, weak, or noisy.

In [ ]:
top_numeric = ["Attendance", "Hours_Studied", "Class_Participation_Score", "Previous_Scores"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

xlabels = {
    "Attendance": "Attendance (%)",
    "Hours_Studied": "Hours studied",
    "Class_Participation_Score": "Class participation score (points)",
    "Previous_Scores": "Previous score (points out of 100)"
}

for ax, col in zip(axes, top_numeric):
    sns.regplot(
        data=analysis_df,
        x=col,
        y="Exam_Score",
        scatter_kws={"alpha": 0.25},
        line_kws={"linewidth": 2},
        ax=ax
    )
    ax.set_title(f"{col.replace('_', ' ')} Shows a Positive Relationship With Exam Score")
    ax.set_xlabel(xlabels[col])
    ax.set_ylabel("Exam score (points out of 100)")

plt.tight_layout()
plt.show()


### Observation

The scatterplots make the ranking more concrete:

- `Attendance` shows the clearest upward pattern.
- `Hours_Studied` and `Class_Participation_Score` also rise with score in a visible way.
- `Previous_Scores` is positive too, but the relationship looks weaker and more dispersed.

The practical takeaway is that current academic engagement appears more informative than past performance alone.


In [ ]:
g = sns.lmplot(
    data=analysis_df,
    x="Hours_Studied",
    y="Exam_Score",
    col="Motivation_Level",
    col_order=sorted(analysis_df["Motivation_Level"].dropna().unique()),
    scatter_kws={"alpha": 0.2},
    height=4,
    aspect=1
)
g.set_axis_labels("Hours studied", "Exam score (points out of 100)")
g.fig.subplots_adjust(top=0.82)
g.fig.suptitle("Small Multiples: Study Hours vs Exam Score Across Motivation Levels")
plt.show()


### Small-multiples interpretation

Faceting the same relationship by `Motivation_Level` helps answer whether the study-time pattern is stable across groups.

The positive slope appears in each panel, which suggests the `Hours_Studied` effect is not being driven by only one motivation category. Motivation may still matter, but study time remains directionally consistent across the three groups.


## Group comparisons for categorical features

For categorical variables, boxplots and grouped summaries help us compare score distributions across categories.

We focus on the features that are most relevant to the project theme:
- motivation
- parental involvement
- access to resources
- internet access
- family income
- teacher quality
- peer influence
- learning disabilities
- school type
- gender

In [ ]:
categorical_focus = [
    "Motivation_Level", "Parental_Involvement", "Access_to_Resources",
    "Internet_Access", "Family_Income", "Teacher_Quality",
    "Peer_Influence", "Learning_Disabilities", "School_Type", "Gender",
    "Parental_Education_Level", "Distance_from_Home", "Extracurricular_Activities",
    "Academic_Year"
]

grouped_means = []
for col in categorical_focus:
    temp = analysis_df.groupby(col)["Exam_Score"].agg(["mean", "median", "count"]).reset_index()
    temp.insert(0, "feature", col)
    grouped_means.append(temp)

grouped_means_df = pd.concat(grouped_means, ignore_index=True)
display(grouped_means_df.head(40))


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 16))
plots = [
    "Parental_Involvement", "Access_to_Resources", "Motivation_Level",
    "Teacher_Quality", "Peer_Influence", "Distance_from_Home"
]

for ax, col in zip(axes.flatten(), plots):
    sns.boxplot(data=analysis_df, x=col, y="Exam_Score", ax=ax)
    ax.set_title(f"Exam Score Distribution by {col.replace('_', ' ')}")
    ax.set_xlabel(col.replace('_', ' '))
    ax.set_ylabel("Exam score (points out of 100)")
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
binary_plots = ["Internet_Access", "Learning_Disabilities", "School_Type", "Gender"]

binary_label_map = {
    "Internet_Access": "Has internet access (Yes / No)",
    "Learning_Disabilities": "Learning disabilities reported (Yes / No)",
    "School_Type": "School type (Public / Private)",
    "Gender": "Gender"
}

for ax, col in zip(axes.flatten(), binary_plots):
    sns.boxplot(data=analysis_df, x=col, y="Exam_Score", ax=ax)
    ax.set_title(f"Exam Score Distribution by {binary_label_map[col]}")
    ax.set_xlabel(binary_label_map[col])
    ax.set_ylabel("Exam score (points out of 100)")

plt.tight_layout()
plt.show()


### Visual observations

These plots help convert the grouped summaries into visible score separation.

Several categorical patterns are visible:

- Higher `Parental_Involvement` tends to align with higher scores.
- Higher `Access_to_Resources` also aligns with higher scores.
- `Motivation_Level` and `Teacher_Quality` show smaller but still visible score differences.
- Students with `Internet_Access = Yes` appear to score slightly higher.
- `School_Type` and `Gender` look very similar across groups.
- Students with `Learning_Disabilities = Yes` appear to score somewhat lower on average.


In [ ]:
academic_year_scores = (
    analysis_df.groupby("Academic_Year")["Exam_Score"]
    .agg(["mean", "median", "count"])
    .reset_index()
    .sort_values("Academic_Year")
)
display(academic_year_scores)

plt.figure(figsize=(8, 4))
plt.bar(academic_year_scores["Academic_Year"], academic_year_scores["mean"])
plt.title("Average Exam Score by Academic Year")
plt.xlabel("Academic year")
plt.ylabel("Average exam score (points out of 100)")
plt.tight_layout()
plt.show()


### Academic year observation

This closes the loop on the time-related fields that were screened earlier.

Average score levels are very similar across `Academic_Year`, so there is no strong sign here that one cohort year behaves fundamentally differently from the others. That makes the broader conclusions look more stable across the available time span.


## Hypotheses

Based on the exploration above, we will test the following hypotheses:

1. Students who study more hours tend to earn higher exam scores.
2. Students with higher attendance tend to earn higher exam scores.
3. Higher class participation is associated with higher exam scores.
4. Exam scores differ across levels of parental involvement.
5. Exam scores differ across levels of access to resources.
6. Students with internet access have different exam scores from those without access.
7. Students with reported learning disabilities have different exam scores.
8. Exam scores differ across school types.
9. Exam scores differ by gender.
10. Sleep hours are associated with exam score.


## Statistical testing

### Test choice

- For numeric predictors vs `Exam_Score`, we use **Spearman correlation**.
  - This is robust for monotonic relationships and works well with discrete numeric variables.
- For 3-level categorical variables, we use **Kruskal-Wallis**.
  - This compares score distributions across multiple groups without assuming normality.
- For binary categorical variables, we use **Mann-Whitney U**.
  - This compares two independent groups.

Because the dataset is large, even small differences can become statistically significant.
So we interpret both the p-value and the practical size of the difference.

In [ ]:
# Numeric hypothesis tests
numeric_test_features = ["Hours_Studied", "Attendance", "Sleep_Hours", "Class_Participation_Score", "Previous_Scores", "Tutoring_Sessions"]

numeric_tests = []
for col in numeric_test_features:
    rho, p = stats.spearmanr(analysis_df[col], analysis_df["Exam_Score"])
    numeric_tests.append({
        "feature": col,
        "test": "Spearman correlation",
        "statistic": rho,
        "p_value": p,
        "significant_at_0_05": p < 0.05
    })

numeric_tests_df = pd.DataFrame(numeric_tests).sort_values("statistic", ascending=False)
display(numeric_tests_df)

In [ ]:
# Multi-group categorical tests
multi_group_features = [
    "Parental_Involvement", "Access_to_Resources", "Motivation_Level",
    "Teacher_Quality", "Family_Income", "Peer_Influence",
    "Parental_Education_Level", "Distance_from_Home"
]

multi_group_tests = []
for col in multi_group_features:
    groups = [g["Exam_Score"].values for _, g in analysis_df.groupby(col)]
    H, p = stats.kruskal(*groups)
    means = analysis_df.groupby(col)["Exam_Score"].mean().to_dict()
    multi_group_tests.append({
        "feature": col,
        "test": "Kruskal-Wallis",
        "statistic": H,
        "p_value": p,
        "significant_at_0_05": p < 0.05,
        "group_means": means
    })

multi_group_tests_df = pd.DataFrame(multi_group_tests).sort_values("p_value")
display(multi_group_tests_df[["feature", "test", "statistic", "p_value", "significant_at_0_05"]])

In [ ]:
# Binary group tests
binary_features = [
    "Internet_Access", "Learning_Disabilities",
    "School_Type", "Gender", "Extracurricular_Activities"
]

binary_tests = []
for col in binary_features:
    if col not in analysis_df.columns:
        continue
    levels = analysis_df[col].dropna().unique().tolist()
    if len(levels) != 2:
        print(f"Skipping {col}: expected 2 levels, found {len(levels)}")
        continue
    group1 = analysis_df.loc[analysis_df[col] == levels[0], "Exam_Score"]
    group2 = analysis_df.loc[analysis_df[col] == levels[1], "Exam_Score"]
    U, p = stats.mannwhitneyu(group1, group2, alternative="two-sided")
    means = analysis_df.groupby(col)["Exam_Score"].mean().to_dict()
    binary_tests.append({
        "feature": col,
        "test": "Mann-Whitney U",
        "statistic": U,
        "p_value": p,
        "significant_at_0_05": p < 0.05,
        "group_means": means
    })

binary_tests_df = pd.DataFrame(binary_tests).sort_values("p_value")
display(binary_tests_df[["feature", "test", "statistic", "p_value", "significant_at_0_05"]])


## Hypothesis results and interpretation

In [ ]:
hypothesis_map = pd.DataFrame([
    ["Hours_Studied",             "More study hours are associated with higher exam scores",                    "Spearman correlation", "rho"],
    ["Attendance",                "Higher attendance is associated with higher exam scores",                   "Spearman correlation", "rho"],
    ["Class_Participation_Score", "Higher class participation is associated with higher exam scores",          "Spearman correlation", "rho"],
    ["Sleep_Hours",               "Sleep hours are associated with exam score",                                "Spearman correlation", "rho"],
    ["Parental_Involvement",      "Exam scores differ across parental involvement levels",                     "Kruskal-Wallis",       "H"],
    ["Access_to_Resources",       "Exam scores differ across access-to-resources levels",                     "Kruskal-Wallis",       "H"],
    ["Internet_Access",           "Students with internet access have different exam scores",                  "Mann-Whitney U",       "U"],
    ["Learning_Disabilities",     "Students with learning disabilities have different exam scores",            "Mann-Whitney U",       "U"],
    ["School_Type",               "Public and private school students have different exam scores",             "Mann-Whitney U",       "U"],
    ["Gender",                    "Male and female students have different exam scores",                       "Mann-Whitney U",       "U"],
], columns=["feature", "hypothesis", "test", "stat_label"])

test_results_long = pd.concat([
    numeric_tests_df[["feature", "test", "statistic", "p_value"]],
    multi_group_tests_df[["feature", "test", "statistic", "p_value"]],
    binary_tests_df[["feature", "test", "statistic", "p_value"]]
], ignore_index=True)

hypothesis_summary = (
    hypothesis_map
    .merge(test_results_long, on=["feature", "test"], how="left")
    .assign(
        evidence=lambda d: d["stat_label"] + " = " + d["statistic"].round(3).astype(str),
        decision=lambda d: np.where(d["p_value"] < 0.05, "Supported", "Not supported")
    )[["hypothesis", "test", "evidence", "p_value", "decision"]]
)

display(hypothesis_summary.sort_values(["decision", "p_value"]))


### Interpretation of hypothesis tests

The strongest supported hypotheses are the academic engagement variables:

- **Attendance** has the clearest relationship with exam performance.
- **Hours_Studied** is also strongly and positively associated with scores.
- **Parental_Involvement** and **Access_to_Resources** show statistically meaningful group differences.

Some variables are statistically significant but practically smaller:
- `Motivation_Level`
- `Teacher_Quality`
- `Family_Income`
- `Peer_Influence`
- `Internet_Access`

Some expected differences are **not** supported here:
- `School_Type`
- `Gender`
- `Sleep_Hours` shows no meaningful association

Together with the outlier check and the stable `Academic_Year` averages, these results suggest the main findings are broad dataset patterns rather than artifacts from a few unusual observations or one specific cohort.


## Final insights

Below are the main insights from the analysis.

1. **Attendance is the strongest non-leakage numeric driver in the dataset.** Students with higher attendance consistently score better, and the correlation is clearly stronger than most other predictors.

2. **Study time matters.** `Hours_Studied` has a strong positive relationship with `Exam_Score`, supporting the idea that sustained effort is linked to better outcomes.

3. **Class engagement matters almost as much as study time.** `Class_Participation_Score` is strongly associated with higher scores, suggesting that active classroom behavior is meaningful, not just time spent studying alone.

4. **Past academic performance still matters, but less strongly than current engagement.** `Previous_Scores` is positively related to exam results, though the effect is weaker than attendance and study habits.

5. **Home support variables matter.** Students with higher `Parental_Involvement`, higher `Parental_Education_Level`, and stronger `Access_to_Resources` tend to perform better on average.

6. **Teacher quality and peer influence show measurable differences.** Students who report higher `Teacher_Quality` and more positive `Peer_Influence` score better on average.

7. **Internet access is associated with slightly higher scores.** The difference is statistically significant, but the score gap is modest, so this looks real but not large.

8. **Learning disabilities are associated with lower exam scores in this dataset.** This does not explain why, but it signals a group that may need stronger academic support.

9. **School type and gender do not create meaningful score differences here.** Public vs private and male vs female distributions are very similar.

10. **Cumulative_GPA should not be used as an explanatory feature.** It is exact target leakage because it is mathematically derived from `Exam_Score`. Including it would make the analysis misleading.

## Conclusion and next steps

This project moved from raw inspection to tested conclusions.

### Key findings
- Academic engagement variables, especially `Attendance`, `Hours_Studied`, and `Class_Participation_Score`, are the clearest factors linked to exam performance.
- Family and environment variables such as `Parental_Involvement`, `Access_to_Resources`, `Teacher_Quality`, and `Peer_Influence` also matter.
- Some variables that might be assumed to matter, such as `School_Type`, `Gender`, and `Sleep_Hours`, do not show strong evidence here.
- Outliers are present but limited, and score patterns remain broadly stable across `Academic_Year`.

### Future work
- Build a clean predictive model **without leakage**
- Add effect size measures for group tests
- Explore interactions such as `Attendance x Motivation_Level`
- Test multivariable models to separate overlapping effects across academic, home, and school factors
